<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/simple_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [223]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader

In [224]:
!ls
file = "./100_Unique_QA_Dataset.csv"

100_Unique_QA_Dataset.csv  sample_data


In [225]:
import pandas as pd

In [226]:
df = pd.read_csv(file)
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [227]:
df.loc[0]

,0
question,What is the capital of France?
answer,Paris


In [228]:
def tokenize(text):
  text = text.lower()
  text = text.replace("?" , "")
  text = text.replace("'" , "")
  return text.split()

In [229]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [230]:
def build_vocab(row):
  token_ques = tokenize(row["question"])
  token_ans = tokenize(row['answer'])

  merge_token = token_ans+token_ques
  # print(merge_token)

  for token in merge_token:
    if token in vocab :
      pass
    else :
      vocab[token] = len(vocab)


In [231]:
vocab = {"<unk>":0}


In [232]:
df.apply(build_vocab , axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [233]:
len(vocab)

324

In [234]:
def text_to_index(text , vocab):
  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else :
      indexed_text.append(vocab["<unk>"])
  return indexed_text

In [235]:
ind = text_to_index("What is the capital of France?" , vocab)
print(ind)

[2, 3, 4, 5, 6, 7]


In [236]:
class QA_dataset(Dataset):
  def __init__(self , df , vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self , index):
    ques = text_to_index(self.df.iloc[index]["question"], self.vocab)
    ans = text_to_index(self.df.iloc[index]["answer"], self.vocab)
    return torch.tensor(ques) , torch.tensor(ans)


In [237]:
dataset = QA_dataset(df , vocab)
dataset[0]

(tensor([2, 3, 4, 5, 6, 7]), tensor([1]))

In [238]:
loaded_data = DataLoader(dataset , batch_size=1 , shuffle=True)

In [239]:
class simpleRNN(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size , 50 )
    self.rnn = nn.RNN(50 , 64 , batch_first=True)
    self.fc = nn.Linear(64 , vocab_size)
  def forward(self , x):
    em = self.embedding(x)
    rnn = self.rnn(em)
    y_hat = self.fc(rnn[1])

    return y_hat

In [240]:
model = simpleRNN(len(vocab))

In [241]:
learning_rate = 0.001
epoch = 200

In [242]:
optimizer = optim.Adam(model.parameters() , lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

In [243]:
for epoch in range(epoch):
  total_loss = 0

  for i , (input , target) in enumerate(loaded_data):
    optimizer.zero_grad()

    output = model(input)
    # print(output.shape)

    output = output.squeeze(1)
    target = target.squeeze(1)
    loss = loss_fn(output , target)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 522.943436
Epoch: 2, Loss: 455.698061
Epoch: 3, Loss: 378.254727
Epoch: 4, Loss: 313.106366
Epoch: 5, Loss: 259.696373
Epoch: 6, Loss: 211.227808
Epoch: 7, Loss: 167.723191
Epoch: 8, Loss: 129.921277
Epoch: 9, Loss: 99.715418
Epoch: 10, Loss: 76.428889
Epoch: 11, Loss: 58.681265
Epoch: 12, Loss: 45.941987
Epoch: 13, Loss: 36.623060
Epoch: 14, Loss: 29.593614
Epoch: 15, Loss: 24.333709
Epoch: 16, Loss: 20.531407
Epoch: 17, Loss: 17.188095
Epoch: 18, Loss: 14.614212
Epoch: 19, Loss: 12.556836
Epoch: 20, Loss: 10.897392
Epoch: 21, Loss: 9.535253
Epoch: 22, Loss: 8.449357
Epoch: 23, Loss: 7.488062
Epoch: 24, Loss: 6.668076
Epoch: 25, Loss: 5.982523
Epoch: 26, Loss: 5.407035
Epoch: 27, Loss: 4.891356
Epoch: 28, Loss: 4.450619
Epoch: 29, Loss: 4.049829
Epoch: 30, Loss: 3.726102
Epoch: 31, Loss: 3.424234
Epoch: 32, Loss: 3.144903
Epoch: 33, Loss: 2.911836
Epoch: 34, Loss: 2.691647
Epoch: 35, Loss: 2.496448
Epoch: 36, Loss: 2.314758
Epoch: 37, Loss: 2.157596
Epoch: 38, Loss: 2.

In [247]:
model.eval()
pred = model(text_to_index("Who wrote 'To Kill a Mockingbird'?" , vocab))

TypeError: embedding(): argument 'indices' (position 2) must be Tensor, not list

In [ ]:
print(pred)